# 📊 Water Pump Challenge - Évaluation des Soumissions

Ce notebook permet d'évaluer les soumissions des stagiaires et de générer un score pour le leaderboard.

**Note pour l'encadrant** : Ce notebook utilise le fichier `data/test_labels_secret.csv` qui ne doit **jamais** être partagé avec les stagiaires.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime

## Configuration

In [ ]:
# ==========================================
# CONFIGURATION - À MODIFIER
# ==========================================

# Chemin vers le fichier de soumission du stagiaire
SUBMISSION_FILE = "ma_soumission.csv"

# Nom du stagiaire (pour le leaderboard)
STAGIAIRE_NOM = "Stagiaire"

# ==========================================

## Chargement des Données

In [ ]:
# Charger les labels secrets (ground truth)
ground_truth = pd.read_csv('data/test_labels_secret.csv')

# Charger la soumission
try:
    submission = pd.read_csv(SUBMISSION_FILE)
    print(f"✅ Fichier de soumission chargé : {SUBMISSION_FILE}")
    print(f"   Nombre de prédictions : {len(submission)}")
except FileNotFoundError:
    print(f"❌ Erreur : Le fichier '{SUBMISSION_FILE}' n'existe pas.")
    print("   Veuillez vérifier le chemin du fichier.")
    raise

## Validation du Format

In [ ]:
def validate_submission(submission, ground_truth):
    """Valide le format de la soumission."""
    errors = []
    
    # Vérifier les colonnes
    required_cols = {'id', 'status_group'}
    if not required_cols.issubset(set(submission.columns)):
        missing = required_cols - set(submission.columns)
        errors.append(f"Colonnes manquantes : {missing}")
    
    # Vérifier le nombre de lignes
    if len(submission) != len(ground_truth):
        errors.append(f"Nombre de lignes incorrect : {len(submission)} (attendu : {len(ground_truth)})")
    
    # Vérifier que tous les IDs sont présents
    missing_ids = set(ground_truth['id']) - set(submission['id'])
    if missing_ids:
        errors.append(f"IDs manquants : {len(missing_ids)} IDs")
    
    extra_ids = set(submission['id']) - set(ground_truth['id'])
    if extra_ids:
        errors.append(f"IDs en trop : {len(extra_ids)} IDs")
    
    # Vérifier les valeurs de status_group
    valid_classes = {'functional', 'functional needs repair', 'non functional'}
    invalid_classes = set(submission['status_group'].unique()) - valid_classes
    if invalid_classes:
        errors.append(f"Classes invalides : {invalid_classes}")
    
    return errors

# Valider
errors = validate_submission(submission, ground_truth)

if errors:
    print("❌ Erreurs de validation :")
    for error in errors:
        print(f"   - {error}")
else:
    print("✅ Format de soumission valide !")

## Calcul du Score

In [ ]:
def calculate_score(submission, ground_truth):
    """Calcule le score d'accuracy."""
    # Fusionner sur l'ID pour aligner les prédictions
    merged = ground_truth.merge(submission, on='id', suffixes=('_true', '_pred'))
    
    y_true = merged['status_group_true']
    y_pred = merged['status_group_pred']
    
    accuracy = accuracy_score(y_true, y_pred)
    
    return accuracy, y_true, y_pred

# Calculer le score
accuracy, y_true, y_pred = calculate_score(submission, ground_truth)

print("=" * 50)
print(f"📊 RÉSULTAT POUR : {STAGIAIRE_NOM}")
print("=" * 50)
print(f"\n🎯 ACCURACY : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"\n📅 Date d'évaluation : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 50)

## Rapport Détaillé

In [ ]:
# Rapport de classification
print("\n📋 RAPPORT DE CLASSIFICATION")
print("-" * 50)
print(classification_report(y_true, y_pred))

In [ ]:
# Matrice de confusion
plt.figure(figsize=(10, 8))
classes = ['functional', 'functional needs repair', 'non functional']
cm = confusion_matrix(y_true, y_pred, labels=classes)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=classes, yticklabels=classes)
plt.title(f'Matrice de Confusion - {STAGIAIRE_NOM}\nAccuracy: {accuracy:.4f}')
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution des prédictions vs réalité
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Prédictions
pd.Series(y_pred).value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Distribution des Prédictions')
axes[0].set_xlabel('Classe')
axes[0].set_ylabel('Nombre')
axes[0].tick_params(axis='x', rotation=45)

# Réalité
pd.Series(y_true).value_counts().plot(kind='bar', ax=axes[1], color='forestgreen')
axes[1].set_title('Distribution Réelle')
axes[1].set_xlabel('Classe')
axes[1].set_ylabel('Nombre')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Entrée pour le Leaderboard

In [ ]:
# Générer une ligne pour le leaderboard
leaderboard_entry = {
    'Nom': STAGIAIRE_NOM,
    'Score': f"{accuracy:.4f}",
    'Accuracy (%)': f"{accuracy*100:.2f}%",
    'Date': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'Fichier': SUBMISSION_FILE
}

print("\n📋 ENTRÉE POUR LE LEADERBOARD :")
print("-" * 50)
for key, value in leaderboard_entry.items():
    print(f"{key}: {value}")

In [ ]:
# Sauvegarder les résultats (optionnel)
import os

# Créer le dossier results s'il n'existe pas
os.makedirs('results', exist_ok=True)

# Sauvegarder le résultat
result_file = f"results/{STAGIAIRE_NOM}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
with open(result_file, 'w') as f:
    f.write(f"Nom: {STAGIAIRE_NOM}\n")
    f.write(f"Score: {accuracy:.4f}\n")
    f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Fichier soumis: {SUBMISSION_FILE}\n")
    f.write(f"\n{classification_report(y_true, y_pred)}")

print(f"\n✅ Résultats sauvegardés dans : {result_file}")

---

## Baseline Scores (pour référence)

| Approche | Score Attendu |
|----------|---------------|
| Prédire toujours "functional" | ~54% |
| Random Forest (features numériques) | ~65-70% |
| Bon modèle avec feature engineering | ~75-80% |
| Excellent modèle optimisé | >80% |